Final Project – Business Sales Analysis (Zara Clothing)

Project Goal
The goal of this project is to analyze a clothing sales dataset from Zara and build a model that predicts **Sales Volume** for each product based on:
- product attributes (price, material, season, type, etc.),
- merchandising decisions (product position in store, promotion, seasonal flag),
- customer segment (section: WOMAN / MAN).

The project follows the full data science lifecycle:
1. **Project planning** – define goal, questions, and approach.
2. **Data acquisition** – load the dataset from Kaggle.
3. **Data processing & cleaning** – parse raw data, fix types, check missing values.
4. **Exploratory data analysis (EDA)** – understand distributions and relationships.
5. **Feature engineering & modeling** – prepare features, train and evaluate models.
6. **Visualization** – show key patterns and model behavior.
7. **Interpretation** – summarize main insights and model performance.

Main Questions
- Do **promotions** and **End-cap placements** increase sales volume?
- How do **price**, **season**, and **seasonal flag** relate to sales?
- Can we build a reasonably accurate model to **predict sales volume** from product and merchandising features?


Cell 2 — Data Acquisition (Code)

In [1]:
# DATA ACQUISITION: download and load dataset

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)

# 1. Download dataset from Kaggle (cached locally after first time)
path = kagglehub.dataset_download("ayeshaseherr/buisness-sales")
print("Dataset folder:", path)

# 2. Find CSV file
files = os.listdir(path)
print("Files in folder:", files)

csv_path = os.path.join(path, "Business_sales_EDA.csv")

# 3. Load raw CSV
df_raw = pd.read_csv(csv_path)
print("\nRaw shape:", df_raw.shape)
display(df_raw.head())


c:\Users\murad\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset folder: C:\Users\murad\.cache\kagglehub\datasets\ayeshaseherr\buisness-sales\versions\1
Files in folder: ['Business_sales_EDA.csv']

Raw shape: (20252, 1)


,Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin
0,185102;Aisle;Yes;clothing;Yes;1243;Zara;https:...
1,188771;Aisle;Yes;clothing;No;1429;Zara;https:/...
2,180176;End-cap;Yes;clothing;Yes;1168;Zara;http...
3,112917;Aisle;Yes;clothing;No;1348;Zara;https:/...
4,192936;End-cap;Yes;clothing;Yes;1602;Zara;http...


Cell 3 — Data Processing & Cleaning (Code)

In [ ]:
# DATA PROCESSING & CLEANING

# 1) Split the single column into separate columns
df = df_raw.iloc[:, 0].str.split(";", expand=True)

df.columns = [
    "Product ID", "Product Position", "Promotion", "Product Category", "Seasonal",
    "Sales Volume", "brand", "url", "name", "description", "price", "currency",
    "terms", "section", "season", "material", "origin"
]

# 2) Convert data types
df["Product ID"]   = pd.to_numeric(df["Product ID"], errors="coerce")
df["Sales Volume"] = pd.to_numeric(df["Sales Volume"], errors="coerce")

# clean price text and convert to float
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace("€", "", regex=False)
    .str.replace(",", "")
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# 3) Basic cleaning: drop exact duplicate rows (if any)
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Rows before dropping duplicates: {before}")
print(f"Rows after dropping duplicates : {after}")

# 4) Check missing values
print("\nMissing values per column:")
print(df.isna().sum())

# 5) Final clean info
print("\nCleaned dtypes:")
print(df.dtypes)

print("\nCleaned head:")
display(df.head())

# we'll use `df` as the cleaned base datase


Dataset loaded.
Shape (rows, columns): (20252, 1)

First 5 rows:


,Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin
0,185102;Aisle;Yes;clothing;Yes;1243;Zara;https:...
1,188771;Aisle;Yes;clothing;No;1429;Zara;https:/...
2,180176;End-cap;Yes;clothing;Yes;1168;Zara;http...
3,112917;Aisle;Yes;clothing;No;1348;Zara;https:/...
4,192936;End-cap;Yes;clothing;Yes;1602;Zara;http...



Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20252 entries, 0 to 20251
Data columns (total 1 columns):
 #   Column                                                                                                                                                       Non-Null Count  Dtype 
---  ------                                                                                                                                                       --------------  ----- 
 0   Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin  20252 non-null  object
dtypes: object(1)
memory usage: 158.3+ KB
None

Numeric summary:


,Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin
count,20252
unique,20252
top,185102;Aisle;Yes;clothing;Yes;1243;Zara;https:...
freq,1



Categorical summary:


,Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin
count,20252
unique,20252
top,185102;Aisle;Yes;clothing;Yes;1243;Zara;https:...
freq,1



Missing values per column:
Product ID;Product Position;Promotion;Product Category;Seasonal;Sales Volume;brand;url;name;description;price;currency;terms;section;season;material;origin    0
dtype: int64


In [ ]:
# DATA PROCESSING & CLEANING

# 1) Split the single column into separate columns
df = df_raw.iloc[:, 0].str.split(";", expand=True)

df.columns = [
    "Product ID", "Product Position", "Promotion", "Product Category", "Seasonal",
    "Sales Volume", "brand", "url", "name", "description", "price", "currency",
    "terms", "section", "season", "material", "origin"
]

# 2) Convert data types
df["Product ID"]   = pd.to_numeric(df["Product ID"], errors="coerce")
df["Sales Volume"] = pd.to_numeric(df["Sales Volume"], errors="coerce")

# clean price text and convert to float
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace("€", "", regex=False)
    .str.replace(",", "")
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# 3) Basic cleaning: drop exact duplicate rows (if any)
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Rows before dropping duplicates: {before}")
print(f"Rows after dropping duplicates : {after}")

# 4) Check missing values
print("\nMissing values per column:")
print(df.isna().sum())

# 5) Final clean info
print("\nCleaned dtypes:")
print(df.dtypes)

print("\nCleaned head:")
display(df.head())

# we'll use `df` as the cleaned base dataset



,Product ID,Product Position,Promotion,Product Category,Seasonal,Sales Volume,brand,url,name,description,price,currency,terms,section,season,material,origin
0,185102,Aisle,Yes,clothing,Yes,1243,Zara,https://www.zara.com/us/en/basic-puffer-jacket...,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,78.99,USD,jackets,MAN,Winter,Polyester,Brazil
1,188771,Aisle,Yes,clothing,No,1429,Zara,https://www.zara.com/us/en/tuxedo-jacket-p0889...,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,14.99,USD,jackets,MAN,Autumn,Cotton,Turkey
2,180176,End-cap,Yes,clothing,Yes,1168,Zara,https://www.zara.com/us/en/slim-fit-suit-jacke...,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,71.95,USD,jackets,WOMAN,Autumn,Polyester,Morocco
3,112917,Aisle,Yes,clothing,No,1348,Zara,https://www.zara.com/us/en/stretch-suit-jacket...,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,30.99,USD,jackets,MAN,Spring,Polyester,China
4,192936,End-cap,Yes,clothing,Yes,1602,Zara,https://www.zara.com/us/en/double-faced-jacket...,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,22.99,USD,jackets,WOMAN,Winter,Wool Blend,China


In [11]:
df_clean.info()
df_clean.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20252 entries, 0 to 20251
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Product ID        20252 non-null  object
 1   Product Position  20252 non-null  object
 2   Promotion         20252 non-null  object
 3   Product Category  20252 non-null  object
 4   Seasonal          20252 non-null  object
 5   Sales Volume      20252 non-null  object
 6   brand             20252 non-null  object
 7   url               20252 non-null  object
 8   name              20252 non-null  object
 9   description       20252 non-null  object
 10  price             20252 non-null  object
 11  currency          20252 non-null  object
 12  terms             20252 non-null  object
 13  section           20252 non-null  object
 14  season            20252 non-null  object
 15  material          20252 non-null  object
 16  origin            20252 non-null  object
dtypes: object(17

(20252, 17)

Convert columns to correct data types

In [12]:
# Copy df
df2 = df_clean.copy()

# Convert numeric columns
df2["Product ID"] = pd.to_numeric(df2["Product ID"], errors="coerce")
df2["Sales Volume"] = pd.to_numeric(df2["Sales Volume"], errors="coerce")

# Clean price: remove currency symbols if any
df2["price"] = (
    df2["price"]
    .str.replace("$", "", regex=False)
    .str.replace("€", "", regex=False)
    .str.replace(",", "")
)
df2["price"] = pd.to_numeric(df2["price"], errors="coerce")

# Check results
df2.info()
df2.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20252 entries, 0 to 20251
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Product ID        20252 non-null  int64  
 1   Product Position  20252 non-null  object 
 2   Promotion         20252 non-null  object 
 3   Product Category  20252 non-null  object 
 4   Seasonal          20252 non-null  object 
 5   Sales Volume      20252 non-null  int64  
 6   brand             20252 non-null  object 
 7   url               20252 non-null  object 
 8   name              20252 non-null  object 
 9   description       20252 non-null  object 
 10  price             20252 non-null  float64
 11  currency          20252 non-null  object 
 12  terms             20252 non-null  object 
 13  section           20252 non-null  object 
 14  season            20252 non-null  object 
 15  material          20252 non-null  object 
 16  origin            20252 non-null  object

,Product ID,Product Position,Promotion,Product Category,Seasonal,Sales Volume,brand,url,name,description,price,currency,terms,section,season,material,origin
0,185102,Aisle,Yes,clothing,Yes,1243,Zara,https://www.zara.com/us/en/basic-puffer-jacket...,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,78.99,USD,jackets,MAN,Winter,Polyester,Brazil
1,188771,Aisle,Yes,clothing,No,1429,Zara,https://www.zara.com/us/en/tuxedo-jacket-p0889...,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,14.99,USD,jackets,MAN,Autumn,Cotton,Turkey
2,180176,End-cap,Yes,clothing,Yes,1168,Zara,https://www.zara.com/us/en/slim-fit-suit-jacke...,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,71.95,USD,jackets,WOMAN,Autumn,Polyester,Morocco
3,112917,Aisle,Yes,clothing,No,1348,Zara,https://www.zara.com/us/en/stretch-suit-jacket...,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,30.99,USD,jackets,MAN,Spring,Polyester,China
4,192936,End-cap,Yes,clothing,Yes,1602,Zara,https://www.zara.com/us/en/double-faced-jacket...,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,22.99,USD,jackets,WOMAN,Winter,Wool Blend,China


✅ CELL 1 — Basic Data Exploration

In [13]:
# BASIC DATA EXPLORATION

print("Dataset Shape:", df2.shape)
print("\nColumn Names:\n", df2.columns.tolist())

print("\nData Types:\n")
print(df2.dtypes)

print("\nMissing Values:\n")
print(df2.isna().sum())

print("\nNumber of Unique Values Per Column:\n")
print(df2.nunique())

print("\nFirst Five Rows:")
display(df2.head())

print("\nCategorical Summary:")
display(df2.describe(include='object').T)

print("\nNumeric Summary:")
display(df2.describe(include='number').T)


Dataset Shape: (20252, 17)

Column Names:
 ['Product ID', 'Product Position', 'Promotion', 'Product Category', 'Seasonal', 'Sales Volume', 'brand', 'url', 'name', 'description', 'price', 'currency', 'terms', 'section', 'season', 'material', 'origin']

Data Types:

Product ID            int64
Product Position     object
Promotion            object
Product Category     object
Seasonal             object
Sales Volume          int64
brand                object
url                  object
name                 object
description          object
price               float64
currency             object
terms                object
section              object
season               object
material             object
origin               object
dtype: object

Missing Values:

Product ID          0
Product Position    0
Promotion           0
Product Category    0
Seasonal            0
Sales Volume        0
brand               0
url                 0
name                0
description         0
price  

,Product ID,Product Position,Promotion,Product Category,Seasonal,Sales Volume,brand,url,name,description,price,currency,terms,section,season,material,origin
0,185102,Aisle,Yes,clothing,Yes,1243,Zara,https://www.zara.com/us/en/basic-puffer-jacket...,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,78.99,USD,jackets,MAN,Winter,Polyester,Brazil
1,188771,Aisle,Yes,clothing,No,1429,Zara,https://www.zara.com/us/en/tuxedo-jacket-p0889...,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,14.99,USD,jackets,MAN,Autumn,Cotton,Turkey
2,180176,End-cap,Yes,clothing,Yes,1168,Zara,https://www.zara.com/us/en/slim-fit-suit-jacke...,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,71.95,USD,jackets,WOMAN,Autumn,Polyester,Morocco
3,112917,Aisle,Yes,clothing,No,1348,Zara,https://www.zara.com/us/en/stretch-suit-jacket...,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,30.99,USD,jackets,MAN,Spring,Polyester,China
4,192936,End-cap,Yes,clothing,Yes,1602,Zara,https://www.zara.com/us/en/double-faced-jacket...,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,22.99,USD,jackets,WOMAN,Winter,Wool Blend,China



Categorical Summary:


,count,unique,top,freq
Product Position,20252,3,Aisle,7810
Promotion,20252,2,No,11812
Product Category,20252,1,clothing,20252
Seasonal,20252,2,No,10136
brand,20252,1,Zara,20252
url,20252,228,https://www.zara.com/us/en/knit-sweater-with-r...,187
name,20252,17216,PLAID OVERSHIRT,8
description,20252,222,Varsity jacket with elastic collar and long sl...,333
currency,20252,1,USD,20252
terms,20252,5,jackets,11232



Numeric Summary:


,count,mean,std,min,25%,50%,75%,max
Product ID,20252.0,208931.432303,8961.076507,110075.0,204442.75,209505.50,214568.25,219631.00
Sales Volume,20252.0,1097.400454,298.234609,518.0,849.00,990.00,1364.25,1940.00
price,20252.0,41.949061,23.380960,12.0,23.95,35.95,53.95,134.99


Hypotheses
1. Products placed on **End-cap** positions have higher sales than products placed in regular aisles.  
2. **Seasonal** items generate higher sales volume during relevant seasons.  
3. Products with **Promotion = Yes** have higher sales compared to non-promoted products.  
4. Higher **price** negatively correlates with sales volume.

Analysis Plan
1. Explore the distribution of key variables (price, sales volume, promotion, section, terms).  
2. Analyze relationships between features and Sales Volume.  
3. Encode categorical variables (Product Position, Promotion, Seasonal, terms, season).  
4. Build predictive models to estimate Sales Volume:
   - Linear Regression  
   - Random Forest  
   - Gradient Boosting  
5. Evaluate models using MAE, RMSE, and R².  
6. Create visualizations (distributions, correlations, feature importance).

Expected Results
- Promotions and End-cap positions will show higher sales.  
- Seasonal products will sell more during matching seasons.  
- Lower-priced items will have higher sales volumes.  

Potential Risks
- Dataset only has Zara products → limited generalization.  
- Descriptions and names may be noisy text.  
- All products belong to one category (clothing).  


✅ CELL 3 — Feature Processing

In [15]:
# FEATURE PROCESSING

import pandas as pd

df3 = df2.copy()

# Encode binary Yes/No columns
binary_cols = ["Promotion", "Seasonal"]
for col in binary_cols:
    df3[col] = df3[col].map({"Yes": 1, "No": 0})

# One-hot encode categorical variables
cat_cols = ["Product Position", "terms", "section", "season", "material", "origin"]
df3 = pd.get_dummies(df3, columns=cat_cols, drop_first=True)

# Drop non-useful columns for modeling
df3 = df3.drop(columns=["url", "name", "description", "brand", "Product Category"])

print("Processed shape:", df3.shape)
df3.head()


Processed shape: (20252, 37)


,Product ID,Promotion,Seasonal,Sales Volume,price,currency,Product Position_End-cap,Product Position_Front of Store,terms_jeans,terms_shoes,...,origin_Brazil,origin_Cambodia,origin_China,origin_India,origin_Morocco,origin_Pakistan,origin_Portugal,origin_Spain,origin_Turkey,origin_Vietnam
0,185102,1,1,1243,78.99,USD,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1,188771,1,0,1429,14.99,USD,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,180176,1,1,1168,71.95,USD,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,112917,1,0,1348,30.99,USD,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
4,192936,1,1,1602,22.99,USD,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False


✅ CELL 4 — Train Models and Evaluate

In [16]:
# MODEL TRAINING AND EVALUATION

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np

# Prepare data
X = df3.drop(columns=["Sales Volume"])
y = df3["Sales Volume"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append((name, mae, rmse, r2))

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R²"])
results_df.sort_values("R²", ascending=False)


ValueError: could not convert string to float: 'USD'